In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- 1. 加载和准备数据 ---
file_path = 'data/capacity_retain.csv'
try:
    df = pd.read_csv(file_path, skiprows=[1])
except FileNotFoundError:
    print(f"错误：找不到文件 '{file_path}'。")
    exit()
except Exception:
    df = pd.read_csv(file_path, skiprows=[1], encoding='gbk')

df.columns = df.columns.str.strip()
df = df.dropna(axis=1, how='all')

# --- 2. 分离自变量 (X) 和因变量 (Y) ---
target_name = df.columns[-1]
features_names = df.columns[1:-1].tolist()
X = df[features_names]
y = df[target_name]

if df.columns[0] == 'Rename':
    df_with_index = df.set_index('Rename')
    y.index = df_with_index.index

# --- 3. 步骤一：获取PC1和PC2的具体构成公式 ---
# 先对数据进行标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# 再进行PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("--- 主成分PC1和PC2的构成公式 ---")
# pca.components_ 的每一行是一个主成分，每一列是对应原始变量的系数（载荷）
pc1_coeffs = pca.components_[0]
pc2_coeffs = pca.components_[1]

pc1_formula = "PC1 ="
for i, name in enumerate(features_names):
    pc1_formula += f" ({pc1_coeffs[i]:.4f}) * scaled({name}) +"
pc1_formula = pc1_formula.rstrip(' +').replace("+ (-", "- (")

pc2_formula = "PC2 ="
for i, name in enumerate(features_names):
    pc2_formula += f" ({pc2_coeffs[i]:.4f}) * scaled({name}) +"
pc2_formula = pc2_formula.rstrip(' +').replace("+ (-", "- (")

print(pc1_formula)
print(pc2_formula)
print("\n注意：上述公式中的 scaled(变量名) 指的是原始变量经过标准化处理后的值。")
print("-" * 50)

# --- 4. 步骤二：获取基于PC1和PC2的拟合方程 ---
polynomial_degree = 2
poly_features = PolynomialFeatures(degree=polynomial_degree, include_bias=False)
X_poly = poly_features.fit_transform(X_pca)
model = LinearRegression()
model.fit(X_poly, y)

intercept = model.intercept_
coeffs = model.coef_
poly_feature_names = poly_features.get_feature_names_out(['PC1', 'PC2'])

equation = f"{target_name} = {intercept:.4f}"
for coeff, name in zip(coeffs, poly_feature_names):
    equation += f" + ({coeff:.4f}) * {name}"
equation = equation.replace("+ (-", "- (")

print("--- 基于PC1和PC2的二次多项式拟合方程 ---")
print(equation)
print("-" * 50)

# --- 5. 步骤三：计算并显示模型误差 ---
y_pred = model.predict(X_poly)
residuals = y - y_pred
error_df = pd.DataFrame({
    '真实值 (Actual)': y,
    '模型预测值 (Predicted)': y_pred,
    '误差 (Error)': residuals
}).round(4)

print("--- 每个样本点的拟合误差详情 ---")
print(error_df)
print("-" * 40)

mse = mean_squared_error(y, y_pred)
mae = mean_absolute_error(y, y_pred)
r2 = r2_score(y, y_pred)

print("--- 模型整体误差评估 ---")
print(f"均方误差 (MSE): {mse:.4f}")
print(f"平均绝对误差 (MAE): {mae:.4f}")
print(f"R² 分数: {r2:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- 1. 加载和准备数据 ---
file_path = 'data/capacity.csv'
try:
    df = pd.read_csv(file_path, skiprows=[1])
except FileNotFoundError:
    print(f"错误：找不到文件 '{file_path}'。")
    exit()
except Exception:
    df = pd.read_csv(file_path, skiprows=[1], encoding='gbk')

df.columns = df.columns.str.strip()
df = df.dropna(axis=1, how='all')

# --- 2. 分离自变量 (X) 和因变量 (Y) ---
target_name = df.columns[-1]
features_names = df.columns[1:-1].tolist()
X = df[features_names]
y = df[target_name]

if df.columns[0] == 'Rename':
    df_with_index = df.set_index('Rename')
    y.index = df_with_index.index

# --- 3. 步骤一：获取PC1和PC2的具体构成公式 ---
# 先对数据进行标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# 再进行PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("--- 主成分PC1和PC2的构成公式 ---")
# pca.components_ 的每一行是一个主成分，每一列是对应原始变量的系数（载荷）
pc1_coeffs = pca.components_[0]
pc2_coeffs = pca.components_[1]

pc1_formula = "PC1 ="
for i, name in enumerate(features_names):
    pc1_formula += f" ({pc1_coeffs[i]:.4f}) * scaled({name}) +"
pc1_formula = pc1_formula.rstrip(' +').replace("+ (-", "- (")

pc2_formula = "PC2 ="
for i, name in enumerate(features_names):
    pc2_formula += f" ({pc2_coeffs[i]:.4f}) * scaled({name}) +"
pc2_formula = pc2_formula.rstrip(' +').replace("+ (-", "- (")

print(pc1_formula)
print(pc2_formula)
print("\n注意：上述公式中的 scaled(变量名) 指的是原始变量经过标准化处理后的值。")
print("-" * 50)

# --- 4. 步骤二：获取基于PC1和PC2的拟合方程 ---
polynomial_degree = 2
poly_features = PolynomialFeatures(degree=polynomial_degree, include_bias=False)
X_poly = poly_features.fit_transform(X_pca)
model = LinearRegression()
model.fit(X_poly, y)

intercept = model.intercept_
coeffs = model.coef_
poly_feature_names = poly_features.get_feature_names_out(['PC1', 'PC2'])

equation = f"{target_name} = {intercept:.4f}"
for coeff, name in zip(coeffs, poly_feature_names):
    equation += f" + ({coeff:.4f}) * {name}"
equation = equation.replace("+ (-", "- (")

print("--- 基于PC1和PC2的二次多项式拟合方程 ---")
print(equation)
print("-" * 50)

# --- 5. 步骤三：计算并显示模型误差 ---
y_pred = model.predict(X_poly)
residuals = y - y_pred
error_df = pd.DataFrame({
    '真实值 (Actual)': y,
    '模型预测值 (Predicted)': y_pred,
    '误差 (Error)': residuals
}).round(4)

print("--- 每个样本点的拟合误差详情 ---")
print(error_df)
print("-" * 40)

mse = mean_squared_error(y, y_pred)
mae = mean_absolute_error(y, y_pred)
r2 = r2_score(y, y_pred)

print("--- 模型整体误差评估 ---")
print(f"均方误差 (MSE): {mse:.4f}")
print(f"平均绝对误差 (MAE): {mae:.4f}")
print(f"R² 分数: {r2:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ============================================================
# 提取 PCA 的 PC1 / PC2 计算方程
#   形式A：标准化变量形式（scaled 变量）
#   形式B：原始变量直接代入形式（把标准化折叠进系数，可直接用原始数值）
# ============================================================

# --- 1. 加载和准备数据 ---
file_path = 'data/capacity and capacity_retain.csv'   # 如需更换数据集，改这里即可
try:
    df = pd.read_csv(file_path, skiprows=[1])
except FileNotFoundError:
    print(f"错误：找不到文件 '{file_path}'。")
    raise
except Exception:
    df = pd.read_csv(file_path, skiprows=[1], encoding='gbk')

df.columns = df.columns.str.strip()
df = df.dropna(axis=1, how='all')

# --- 2. 分离自变量 (X) ---
features_names = df.columns[1:-2].tolist()
X = df[features_names]

# --- 3. 标准化 + PCA(取前2个主成分) ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# 载荷：pca.components_ 每一行是一个主成分，每一列对应一个原始变量
loadings = pca.components_          # shape = (2, n_features)
means = scaler.mean_               # 每个变量的均值
stds = scaler.scale_               # 每个变量的标准差(总体std, ddof=0)

# ============================================================
# 形式A：标准化变量形式  PC = Σ 载荷_i * scaled(x_i)
#        其中 scaled(x_i) = (x_i - mean_i) / std_i
# ============================================================
def build_scaled_formula(coeffs, name):
    s = f"{name} ="
    for c, fn in zip(coeffs, features_names):
        s += f" ({c:.4f}) * scaled({fn}) +"
    return s.rstrip(' +').replace("+ (-", "- (")

pc1_scaled_formula = build_scaled_formula(loadings[0], "PC1")
pc2_scaled_formula = build_scaled_formula(loadings[1], "PC2")

# ============================================================
# 形式B：原始变量直接代入形式  PC = b0 + Σ b_i * x_i
#   把标准化折叠进系数：
#     b_i = 载荷_i / std_i
#     b0  = -Σ (载荷_i * mean_i / std_i)
#   可直接代入原始(未标准化)数值计算，无需再查均值/标准差
# ============================================================
raw_coef = loadings / stds                          # shape = (2, n_features)
raw_intercept = -(raw_coef * means).sum(axis=1)     # shape = (2,)

def build_raw_formula(coeffs, b0, name):
    s = f"{name} = {b0:.6f}"
    for c, fn in zip(coeffs, features_names):
        s += f" + ({c:.6f}) * {fn}"
    return s.replace("+ (-", "- (")

pc1_raw_formula = build_raw_formula(raw_coef[0], raw_intercept[0], "PC1")
pc2_raw_formula = build_raw_formula(raw_coef[1], raw_intercept[1], "PC2")

# --- 打印两种形式 ---
print("=" * 60)
print("【形式A】标准化变量形式  scaled(x) = (x - mean) / std")
print("=" * 60)
print(pc1_scaled_formula)
print(pc2_scaled_formula)
print()
print("=" * 60)
print("【形式B】原始变量直接代入形式（可直接用原始数值计算）")
print("=" * 60)
print(pc1_raw_formula)
print(pc2_raw_formula)

# --- 校验：用原始变量形式手算的 PC 与 sklearn 输出是否一致 ---
pc_manual = X.values @ raw_coef.T + raw_intercept
max_diff = np.abs(pc_manual - X_pca).max()
print(f"\n[校验] 原始变量公式手算结果与 sklearn 输出最大偏差: {max_diff:.2e}  (应≈0)")

# ============================================================
# 保存到文件，方便后续直接调用
# ============================================================
coef_df = pd.DataFrame({
    '变量': features_names,
    'PC1_载荷(标准化系数)': loadings[0],
    'PC2_载荷(标准化系数)': loadings[1],
    '均值mean': means,
    '标准差std': stds,
    'PC1_原始变量系数': raw_coef[0],
    'PC2_原始变量系数': raw_coef[1],
})
intercept_row = pd.DataFrame({
    '变量': ['截距(Intercept)'],
    'PC1_载荷(标准化系数)': [np.nan],
    'PC2_载荷(标准化系数)': [np.nan],
    '均值mean': [np.nan],
    '标准差std': [np.nan],
    'PC1_原始变量系数': [raw_intercept[0]],
    'PC2_原始变量系数': [raw_intercept[1]],
})
coef_out = pd.concat([coef_df, intercept_row], ignore_index=True)
coef_csv = 'pc1_pc2_equations_coeffs.csv'
coef_out.to_csv(coef_csv, index=False, encoding='utf-8-sig')

txt_path = 'pc1_pc2_equations.txt'
with open(txt_path, 'w', encoding='utf-8') as f:
    f.write("【形式A】标准化变量形式  scaled(x) = (x - mean) / std\n")
    f.write(pc1_scaled_formula + "\n")
    f.write(pc2_scaled_formula + "\n\n")
    f.write("【形式B】原始变量直接代入形式（可直接用原始数值计算）\n")
    f.write(pc1_raw_formula + "\n")
    f.write(pc2_raw_formula + "\n")

print(f"\n系数表已保存: {coef_csv}")
print(f"方程文本已保存: {txt_path}")
